#Introdução
### Coleta de Dados — RIPE Atlas

Este notebook implementa a primeira etapa do projeto de predição
de falhas de rede.

O objetivo desta etapa é realizar a coleta de dados de medições
de rede utilizando a API do RIPE Atlas.

Os dados coletados serão posteriormente tratados e utilizados
como entrada para um modelo baseado em árvore de decisão.

## Identificação da equipe


- **Equipe:**
- **Integrantes:**
- **Turma:**
- **Data da coleta:**


## Decisões da equipe e Justificativas

Cada equipe deve pesquisar e definir os próprios parâmetros. Registre abaixo:

| Decisão | Escolha da equipe | Justificativa |
|---|---|---|
| ID da medição (`measurement_id`) |  |  |
| Intervalo consultado |  |  |
| Duração do intervalo |  |  |
| Tipo de medição |  |  |
| Formato de saída: CSV ou Parquet |  |  |

Antes de escolher, consultem a documentação e verifiquem se a medição é pública, se o ID existe e se o intervalo solicitado possui resultados. Evitem períodos excessivamente longos: eles podem retornar muitos registros ou tornar a consulta lenta.

[[Justificativas serão inseridas aqui]]


# Roteiro da Primeira Implementação

1. Importação das bibliotecas
2. Configuração da pasta raw
3. Configuração da API
4. Função de coleta de dados
5. Execução da coleta
6. Visualização da resposta da API
7. Função para conversão de JSON em DataFrame
8. Visualização dos dados
9. Salvar os dados na pasta
10. Execução das funções de exportação dos dados
11. Contribuições individuais
12. Checklist da equipe

## Importação das bibliotecas

In [ ]:
import requests
import pandas as pd
import datetime as dt
import json

from pathlib import Path

## Configuração da pasta raw

Deve-se escolher o bloco a ser executado dependendo do ambiente na qual esse notebook estiver sendo executado

In [ ]:
# caso esteja sendo executado no Visual Studio Code, o caminho relativo para a pasta raw será esse
# RAW_FOLDER_PATH = '../raw'

In [ ]:
# caso esteja sendo executado no Google Colab, o caminho de pastas será esse
from google.colab import drive

drive.mount("/content/drive")

PROJECT_FOLDER = Path("/content/drive/MyDrive/ripe_atlas")
RAW_FOLDER_PATH = PROJECT_FOLDER / 'raw'
RAW_FOLDER_PATH.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuração da API

In [ ]:
# métricas definidas pela equipe
MEASUREMENT_ID = 1001
MEASUREMENT_MINUTES_TIME = 3
EXPORTATION_FORMAT = 'csv'
TIMEOUT = 60
PUBLIC_ONLY = True

# gera dois datetimes, sendo um deles com um tempo definido de diferença
# para retornar dados atualizados da API em um intervalo
stop_datetime = dt.datetime.today()
start_datetime = stop_datetime - dt.timedelta(minutes=MEASUREMENT_MINUTES_TIME)

API_URL = f'https://atlas.ripe.net/api/v2/measurements/{MEASUREMENT_ID}/results/'

# sempre coleta dados atualizados, levando em conta a hora atual
START_TIME = start_datetime.isoformat()
STOP_TIME = stop_datetime.isoformat()

# parametros da requisição
params = {
    'start': START_TIME,
    'stop': STOP_TIME,
    'public_only': str(PUBLIC_ONLY).lower()
}

## Função de coleta de dados

In [ ]:
def collect_data(url: str, params: dict, timeout: int) -> dict:
    response = requests.get(
        url,
        params=params,
        timeout=timeout
    )

    response.raise_for_status()
    return response.json()

## Execução da coleta


In [ ]:
request_start_time = dt.datetime.now()
try:
    raw_json_data = collect_data(API_URL, params, TIMEOUT)

except requests.exceptions.Timeout as error:
    raise RuntimeError("Tempo de requisição excedido. A API não respondeu dentro do tempo limite.") from error
except requests.exceptions.HTTPError as error:
    raise RuntimeError(f"Erro na requisição HTTP: {error.response.status_code} - {error.response.reason}") from error
except requests.exceptions.RequestException as error:
    raise RuntimeError(f"Erro na comunicação com a API: {error}") from error
except ValueError as error:
    raise RuntimeError(f"Erro ao processar a resposta da API: {error}") from error
except Exception as error:
    raise RuntimeError(f"Ocorreu um erro inesperado: {error}") from error

if not raw_json_data:
    raise RuntimeError("Nenhum dado foi retornado pela API. Verifique os parâmetros da requisição.")

## Visualização da resposta da API

In [ ]:
print(json.dumps(raw_json_data[0], indent=4))
print(request_start_time)

{
    "fw": 5130,
    "mver": "2.6.4",
    "lts": 10,
    "dst_name": "193.0.14.129",
    "af": 4,
    "dst_addr": "193.0.14.129",
    "src_addr": "192.168.168.2",
    "proto": "ICMP",
    "ttl": 57,
    "size": 32,
    "result": [
        {
            "rtt": 38.031988
        },
        {
            "rtt": 38.054762
        },
        {
            "rtt": 38.156501
        }
    ],
    "dup": 0,
    "rcvd": 3,
    "sent": 3,
    "min": 38.031988,
    "max": 38.156501,
    "avg": 38.081083666666665,
    "msm_id": 1001,
    "prb_id": 1000021,
    "timestamp": 1789693626,
    "msm_name": "Ping",
    "from": "176.9.194.249",
    "type": "ping",
    "step": 240,
    "stored_timestamp": 1789693693
}
2026-09-18 01:10:05.669048


## Função para Conversão de JSON para DataFrame

In [ ]:
def normalize_relevant_data(data: list) -> pd.DataFrame | None:
    if not data:
        return None

    return pd.json_normalize(data, sep='_')


## Visualização dos dados

In [ ]:
raw_df = normalize_relevant_data(raw_json_data)
display(raw_df.head())

,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,max,avg,msm_id,prb_id,timestamp,msm_name,from,type,step,stored_timestamp
0,5130,2.6.4,10,193.0.14.129,4,193.0.14.129,192.168.168.2,ICMP,57.0,32,...,38.156501,38.081084,1001,1000021,1789693626,Ping,176.9.194.249,ping,240,1789693693
1,5080,2.6.2,2242684,193.0.14.129,4,193.0.14.129,192.168.2.200,ICMP,58.0,32,...,18.206638,18.164788,1001,1000023,1789693679,Ping,159.146.31.90,ping,240,1789693757
2,5130,2.6.4,27,193.0.14.129,4,193.0.14.129,192.168.0.14,ICMP,56.0,32,...,125.713496,124.127263,1001,1000054,1789693679,Ping,2.60.190.26,ping,240,1789693715
3,5080,2.6.2,16,193.0.14.129,4,193.0.14.129,89.189.191.217,ICMP,61.0,32,...,0.511989,0.460157,1001,1000055,1789693671,Ping,89.189.191.217,ping,240,1789693711
4,5080,2.6.3,42,193.0.14.129,4,193.0.14.129,188.75.184.76,ICMP,56.0,32,...,8.944798,8.904513,1001,1000069,1789693655,Ping,188.75.184.76,ping,240,1789693730


## Salvar os Dados na Pasta

Arquivos que devem ser salvos:

- Um arquivo .json com a resposta original, sem transformação;
- Um arquivo .csv ou .parquet com o conteúdo exibido no DataFrame;
- Um arquivo de metadados .json contendo as evidências da coleta.

In [ ]:
def export_raw_json(raw_json_data: list, filename: str, path: str):
    if not raw_json_data:
        raise ValueError("Os dados JSON estão vazios. Não há dados para exportar.")

    filename = f'{filename}.json'

    with open(f'{path}/{filename}', 'w') as file:
        json.dump(raw_json_data, file, indent=4)

In [ ]:
def export_raw_df(df_raw: pd.DataFrame, filename: str, format: str, path: str):
    if df_raw is None or df_raw.empty:
        raise ValueError("O DataFrame está vazio. Não há dados para exportar.")

    filename = f'{filename}.{format}'

    if format == 'csv':
        df_raw.to_csv(f'{path}/{filename}', index=False)
    else:
        raise ValueError(f"Formato de exportação '{format}' não suportado.")

In [ ]:
def export_metadata(
        raw_json_data: list,
        raw_df: pd.DataFrame,
        measurement_id: int,
        url: str,
        params: dict,
        start_datetime: dt.datetime,
        stop_datetime: dt.datetime,
        request_start_time: dt.datetime,
        base_filename: str,
        path: str,
        exportation_format: str
):
    metadata = {
        'measurement_id': measurement_id,
        'url': url,
        'params': params,
        'start_datetime': start_datetime.isoformat(),
        'stop_datetime': stop_datetime.isoformat(),
        'request_start_time': request_start_time.isoformat(),
        'total_records': len(raw_json_data),
        'total_columns': len(raw_df.columns),
        'json_file': f'{path}/{base_filename}.json',
        'df_file': f'{path}/{base_filename}.{exportation_format}'
    }

    metadata_filename = f'{path}/{base_filename}_metadata.json'
    with open(metadata_filename, 'w') as file:
        json.dump(metadata, file, indent=4)

## Execução das funções de exportação dos dados

In [ ]:
# com base no horário de execução da requisição cria um timestamp para o nome do arquivo
file_timestamp = request_start_time.strftime("%Y%m%dT%H%M%S")
base_filename = f'ripe_atlas_m{MEASUREMENT_ID}_{file_timestamp}'

export_raw_json(raw_json_data, base_filename, RAW_FOLDER_PATH)
export_raw_df(raw_df, base_filename, EXPORTATION_FORMAT, RAW_FOLDER_PATH)
export_metadata(
    raw_json_data,
    raw_df,
    MEASUREMENT_ID,
    API_URL,
    params,
    start_datetime,
    stop_datetime,
    request_start_time,
    base_filename,
    RAW_FOLDER_PATH,
    EXPORTATION_FORMAT
)



## Contribuições individuais

Integrante 1 — `[Nome completo do aluno]`

* **Atividade realizada:**
  `[Descreva exatamente o que você produziu, pesquisou, analisou, programou, testou ou revisou.]`

* **Parte do trabalho relacionada:**
  `[Informe a seção, arquivo, código, tarefa ou decisão em que você participou.]`

* **Tempo dedicado (aproximado):**
  `[Ex.: 3h30]`

* **Evidência da contribuição:**
  `[Insira o link do repositório dos arquivos que esta usando como evidência]`

* **Explicação da evidência:**
  `[Explique brevemente o que essa evidência comprova e onde sua contribuição pode ser identificada.]`

Integrante 2 — `[Nome completo do aluno]`

* **Atividade realizada:**
  `[Descreva exatamente o que você produziu, pesquisou, analisou, programou, testou ou revisou.]`

* **Parte do trabalho relacionada:**
  `[Informe a seção, arquivo, código, tarefa ou decisão em que você participou.]`

* **Tempo dedicado (aproximado):**
  `[Ex.: 3h30]`

* **Evidência da contribuição:**
  `[Insira o link do repositório dos arquivos que esta usando como evidência]`

* **Explicação da evidência:**
  `[Explique brevemente o que essa evidência comprova e onde sua contribuição pode ser identificada.]`

Integrante 3 — `[Nome completo do aluno]`

* **Atividade realizada:**
  `[Descreva exatamente o que você produziu, pesquisou, analisou, programou, testou ou revisou.]`

* **Parte do trabalho relacionada:**
  `[Informe a seção, arquivo, código, tarefa ou decisão em que você participou.]`

* **Tempo dedicado (aproximado):**
  `[Ex.: 3h30]`

* **Evidência da contribuição:**
  `[Insira o link do repositório dos arquivos que esta usando como evidência]`

* **Explicação da evidência:**
  `[Explique brevemente o que essa evidência comprova e onde sua contribuição pode ser identificada.]`


Integrante 4 — `[Nome completo do aluno]`

* **Atividade realizada:**
  `[Descreva exatamente o que você produziu, pesquisou, analisou, programou, testou ou revisou.]`

* **Parte do trabalho relacionada:**
  `[Informe a seção, arquivo, código, tarefa ou decisão em que você participou.]`

* **Tempo dedicado (aproximado):**
  `[Ex.: 3h30]`

* **Evidência da contribuição:**
  `[Insira o link do repositório dos arquivos que esta usando como evidência]`

* **Explicação da evidência:**
  `[Explique brevemente o que essa evidência comprova e onde sua contribuição pode ser identificada.]`

Integrante 5 — `[Nome completo do aluno]`

* **Atividade realizada:**
  `[Descreva exatamente o que você produziu, pesquisou, analisou, programou, testou ou revisou.]`

* **Parte do trabalho relacionada:**
  `[Informe a seção, arquivo, código, tarefa ou decisão em que você participou.]`

* **Tempo dedicado (aproximado):**
  `[Ex.: 3h30]`

* **Evidência da contribuição:**
  `[Insira o link do repositório dos arquivos que esta usando como evidência]`

* **Explicação da evidência:**
  `[Explique brevemente o que essa evidência comprova e onde sua contribuição pode ser identificada.]`




**Observação**:

Evidências aceitas
1. histórico de edição de documento compartilhado;
2. commit ou pull request no GitHub;
3. arquivo ou trecho de código produzido;
4. relatório, planilha, diagrama, apresentação ou rascunho;
5. registro de tarefa no Trello, GitHub Projects ou ferramenta semelhante;
6. registro de testes realizados;
7. print de reunião, conversa ou e-mail relacionado à atividade;
8. outro material que demonstre claramente a contribuição individual.

Sendo que a evidência deverá identificar o aluno e estar relacionada à atividade declarada. Sempre que possível, apresente materiais com data, autoria ou histórico de edição. Verifique se todos os links estão acessíveis. Trabalhos realizados em conjunto também devem indicar a contribuição específica de cada integrante. Respostas genéricas, como “ajudei no trabalho”, “fiz a pesquisa” ou “participei do código”, não serão consideradas suficientes. Prints de conversas podem complementar a comprovação, mas não devem ser a única evidência quando houver um produto verificável, como código, documento, planilha ou apresentação.
A ausência de descrição ou de evidência poderá impedir a validação da contribuição individual.

## 12. Checklist da equipe

- [ ] Identificamos os integrantes e a turma;
- [ ] Justificamos o ID da medição e o intervalo escolhido;
- [ ] A requisição terminou com status HTTP 200;
- [ ] O JSON retornou pelo menos um registro;
- [ ] Exibimos o `DataFrame` no notebook;
- [ ] Salvamos o JSON original na pasta `raw`;
- [ ] Salvamos o DataFrame em CSV ou Parquet na pasta `raw`;
- [ ] Mantivemos o arquivo de metadados da coleta;
- [ ] Não realizamos limpeza, agregação, rotulação ou treinamento nesta etapa.

### Registro final da equipe

[Escrevam em poucas linhas o que foi coletado, por que esses parâmetros foram escolhidos e qual foi o volume obtido.]
